# 04 — Give Me Some Credit + XGBoost

**Goal:** step up from a clean 1,000-row dataset to a **messy 150,000-row** one, and meet the model
that dominates tabular credit data: **XGBoost**.

You'll learn:
- EDA and cleaning on real, dirty data — severe imbalance, missing values, data-quality errors.
- Why **accuracy is even more useless** here (6.7% default rate).
- How **gradient boosting** works and why it beats the linear scorecard.
- XGBoost's **native missing-value handling** and how to fight imbalance with `scale_pos_weight`.
- **Early stopping**, then **Optuna** tuning to push AUC.

Two cells are `TODO (YOU)`.

## 1. Setup + load

`index_col=0` drops the unnamed row-id column. The target is `SeriousDlqin2yrs`
(1 = borrower became 90+ days delinquent within 2 years = "bad").

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_theme(style="whitegrid")

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import ks_2samp
from xgboost import XGBClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

TARGET = "SeriousDlqin2yrs"
raw = pd.read_csv("../data/raw/cs-training.csv", index_col=0)
print("shape:", raw.shape)
raw.head()

## 2. The imbalance, at scale

German Credit was 30% default. Here it's under 7%. Watch what that does to the accuracy baseline.

In [ ]:
counts = raw[TARGET].value_counts()
rate = raw[TARGET].mean()
print(counts.to_string())
print(f"\ndefault rate = {rate:.4f}  ({rate:.1%})")
print(f"'predict everyone repays' accuracy = {1 - rate:.1%}  <-- useless but looks great")

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["repaid (0)", "default (1)"], counts.values, color=["#2a9d8f", "#e76f51"])
ax.set_title(f"Give Me Some Credit: {rate:.1%} default rate")
ax.set_ylabel("borrowers")
plt.show()

**TODO (YOU):** the imbalance also tells us how to weight the model. XGBoost's `scale_pos_weight`
should be set to `(# negatives) / (# positives)` so the rare defaulters count more. Compute it.

In [ ]:
# TODO (YOU): compute scale_pos_weight = number of repaid / number of defaulted (whole dataset)
scale_pos_weight_full = ...
print("scale_pos_weight (full data) =", round(scale_pos_weight_full, 2), " (expect ~14)")

## 3. Missing values

Two columns have gaps. `MonthlyIncome` is missing ~20% — too much to ignore, too much to trust a
naive fill. We'll let XGBoost handle NaN directly (its native trick), and give logistic regression a
median-imputed version later so it isn't crippled.

In [ ]:
miss = raw.isna().sum()
miss = miss[miss > 0]
print((miss / len(raw)).round(3).to_string(), "\n(fraction missing)")

## 4. Data-quality landmines

Real data has errors that look like values. Three here:
- `age` has a 0 (nobody takes a loan at age 0).
- The three past-due count columns contain **96** and **98** — sentinel codes, not real counts
  (you can't be "98 times 30-59 days late"). They cluster suspiciously.
- `DebtRatio` and `RevolvingUtilizationOfUnsecuredLines` have wild outliers.

In [ ]:
pastdue = ["NumberOfTime30-59DaysPastDueNotWorse",
           "NumberOfTime60-89DaysPastDueNotWorse",
           "NumberOfTimes90DaysLate"]
print("age == 0 rows:", int((raw["age"] == 0).sum()))
print("\nsentinel 96/98 counts in past-due columns:")
for c in pastdue:
    print(f"  {c}: 96 -> {(raw[c]==96).sum()}, 98 -> {(raw[c]==98).sum()}")
print("\ntop of the ratio columns (should be ~0-1, clearly aren't):")
print(raw[["DebtRatio", "RevolvingUtilizationOfUnsecuredLines"]].max().to_string())

## 5. Clean the genuine errors

Minimal, defensible cleaning: turn the impossible `age==0` and the `96/98` sentinels into `NaN`.
We deliberately **do not** clip the big ratio outliers — tree models split on rank order, so a huge
value just lands in the top bucket and doesn't distort anything (unlike a linear model). We also add
`MonthlyIncome_missing` as its own feature, because *whether* income is missing can itself be signal.

In [ ]:
def clean(df):
    df = df.copy()
    df.loc[df["age"] == 0, "age"] = np.nan
    for c in pastdue:
        df.loc[df[c].isin([96, 98]), c] = np.nan
    df["MonthlyIncome_missing"] = df["MonthlyIncome"].isna().astype(int)
    return df

data = clean(raw)
print("cleaned. new column added:", "MonthlyIncome_missing" in data.columns)
print("NaNs XGBoost will handle natively:", int(data.drop(columns=[TARGET]).isna().sum().sum()))

## 6. Quick EDA: which features separate good from bad?

Same "rate by group" move as Phase 1. The past-due history columns are the obvious strong signals;
let's confirm, and look at default rate across age bands.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# default rate by number of times 30-59 days late (capped view)
c = "NumberOfTime30-59DaysPastDueNotWorse"
by_late = data.groupby(data[c].clip(upper=5))[TARGET].mean()
axes[0].bar(by_late.index.astype(str), by_late.values, color="#e76f51")
axes[0].set_title("default rate by # times 30-59 days late (5 = 5+)")
axes[0].set_xlabel("times late"); axes[0].set_ylabel("default rate")

# default rate by age band
age_band = pd.cut(data["age"], bins=[18, 30, 40, 50, 60, 70, 110])
by_age = data.groupby(age_band, observed=True)[TARGET].mean()
axes[1].bar([str(i) for i in by_age.index], by_age.values, color="#264653")
axes[1].set_title("default rate by age band")
axes[1].set_xlabel("age"); axes[1].set_ylabel("default rate")
plt.setp(axes[1].get_xticklabels(), rotation=20)
plt.tight_layout(); plt.show()

Payment history is enormously predictive (default rate climbs steeply with each late episode),
and younger borrowers default more. Exactly the kind of nonlinear, threshold-y structure trees love.

## 7. Train / validation / test split

Three-way split (60/20/20), stratified to preserve the 6.7% rate everywhere. Why three?
- **train**: the model learns on it.
- **validation**: XGBoost watches this during training to know when to stop (early stopping).
- **test**: untouched until the very end — the honest score.

In [ ]:
X = data.drop(columns=[TARGET])
y = data[TARGET]

X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=42)   # 0.25*0.8 = 0.20

for name, yy in [("train", y_tr), ("valid", y_val), ("test", y_test)]:
    print(f"{name}: {len(yy):>6}  default rate {yy.mean():.3f}")

spw = (y_tr == 0).sum() / (y_tr == 1).sum()
print("\nscale_pos_weight (train) =", round(spw, 2))

## 8. How XGBoost works (the 2-minute version)

Boosting builds trees **one at a time**, each new tree trained to predict the *errors* the current
ensemble is still making. Add hundreds of shallow trees, each nudging the prediction, and you get a
flexible model that captures interactions ("young AND high utilization AND recently late") without
you spelling them out.

The knobs that matter most:
- `n_estimators` — how many trees (we cap high and let early stopping pick the real number).
- `learning_rate` — how big each tree's nudge is. Smaller = more trees, usually better, slower.
- `max_depth` — how deep each tree; deeper = more interactions but more overfitting risk.
- `subsample`, `colsample_bytree` — train each tree on a random slice of rows/columns (regularization).
- `scale_pos_weight` — up-weights the rare positives to fight imbalance.
- **early stopping** — stop adding trees once validation AUC stops improving.

## 9. Baseline XGBoost

In [ ]:
base = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=spw,
    eval_metric="auc",
    early_stopping_rounds=50,
    n_jobs=-1,
    random_state=42,
)
base.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
print("best iteration (trees kept):", base.best_iteration)

def report(model, X_te, y_te, label):
    p = model.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_te, p)
    gini = 2 * auc - 1
    ks = ks_2samp(p[y_te == 1], p[y_te == 0]).statistic
    print(f"{label:24s} AUC={auc:.4f}  Gini={gini:.4f}  KS={ks:.4f}")
    return p

_ = report(base, X_test, y_test, "XGBoost (baseline)")

## 10. Logistic regression baseline for contrast

The classical linear model, given a fair shot (median imputation + scaling, since it can't eat NaN).
This is the honest "did boosting actually help?" comparison the brief asks for.

In [ ]:
logit = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=2000, class_weight="balanced"),
)
logit.fit(X_tr, y_tr)
_ = report(logit, X_test, y_test, "Logistic Regression")
_ = report(base,  X_test, y_test, "XGBoost (baseline)")

Expect XGBoost to win by a clear margin on AUC. That gap is why gradient boosting is the default
first move on tabular data, and it's the score deep learning will have to beat later.

## 11. Feature importance

Which features drive the model? XGBoost ranks them by how much each reduces error across all splits.
(This is a global, model-level view; per-applicant explanations via SHAP come in Phase 5.)

In [ ]:
imp = pd.Series(base.feature_importances_, index=X_tr.columns).sort_values()
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(imp.index, imp.values, color="#457b9d")
ax.set_title("XGBoost feature importance (gain)")
plt.tight_layout(); plt.show()

## 12. Tune with Optuna

Optuna searches the hyperparameter space smartly: each trial trains a model with early stopping and
reports validation AUC, and Optuna uses past trials to propose better settings. We run a modest
25 trials (bump `n_trials` up if you want to squeeze more). It optimizes on **validation**, never
touching test.

In [ ]:
def objective(trial):
    params = dict(
        max_depth=trial.suggest_int("max_depth", 3, 8),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
        gamma=trial.suggest_float("gamma", 0.0, 5.0),
    )
    m = XGBClassifier(
        n_estimators=1000, scale_pos_weight=spw, eval_metric="auc",
        early_stopping_rounds=40, n_jobs=-1, random_state=42, **params)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    return roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=25, show_progress_bar=False)
print("best validation AUC:", round(study.best_value, 4))
print("best params:", study.best_params)

Retrain with the best params and score on the untouched test set — the number you'd actually
report.

In [ ]:
tuned = XGBClassifier(
    n_estimators=1000, scale_pos_weight=spw, eval_metric="auc",
    early_stopping_rounds=40, n_jobs=-1, random_state=42, **study.best_params)
tuned.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

_ = report(base,  X_test, y_test, "XGBoost (baseline)")
_ = report(tuned, X_test, y_test, "XGBoost (tuned)")

**TODO (YOU):** does model depth matter? Train one XGBoost with `max_depth=2` (very shallow) and
one with `max_depth=10` (deep), keep everything else the same as the baseline, and compare their test
AUC using `report(...)`. Which overfits, which underfits? (Watch `best_iteration` too — deeper trees
usually need fewer of them.)

In [ ]:
# TODO (YOU): train two models with different max_depth and compare via report(...)
# shallow = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=2, subsample=0.8,
#                         colsample_bytree=0.8, scale_pos_weight=spw, eval_metric="auc",
#                         early_stopping_rounds=50, n_jobs=-1, random_state=42)
# shallow.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
# report(shallow, X_test, y_test, "XGBoost depth=2")
#
# deep = ...  # same but max_depth=10
# report(deep, X_test, y_test, "XGBoost depth=10")

## 13. Recap — what you built

- Handled **real messy data** at 150k scale: severe imbalance, 20% missing income, sentinel codes,
  wild outliers.
- Learned XGBoost's **native NaN handling** (pass the gaps in directly) and why trees **ignore
  monotonic outliers** — so cleaning was minimal and targeted.
- Fought imbalance with **`scale_pos_weight`** and used **early stopping** to pick tree count.
- Saw **XGBoost beat logistic regression** on AUC — the reason it's the tabular default.
- Tuned with **Optuna** and reported the honest **test** AUC (target was >0.75; you cleared it).

**Next session:** round out the classical model bench — **LightGBM and CatBoost** on the same data,
compared head-to-head (AUC/KS/Gini) with XGBoost, plus **probability calibration** so the model's
output is a true probability of default, not just a good ranking.